# 04 · TreeSHAP explanations and robustness

1. **TreeSHAP** (exact, via XGBoost `pred_contribs`): top features per class and an additivity check.
2. **Robustness** of Random Forest / XGBoost / ensemble to three perturbations (same definitions and levels as
   `scripts/evaluate.py`): raw-space multiplicative jitter, Gaussian noise in IQR units, random feature masking.
3. **Explanation stability:** how much the top-5 SHAP features of a flow change under noise and masking.

**Needs:** outputs of notebooks 01 and 02. Perturbation is applied to a random subset of the test split
(`N_ROB`) to keep the run time reasonable; set `N_ROB = None` to use the whole test split.

In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

WORK_DIR = Path(os.environ.get("ASTRA_WORK_DIR", "astra_run"))   # shared by all four notebooks
ART_DIR = WORK_DIR / "artifacts"
SEED = 42
CLASSES = ["benign", "bruteforce", "ddos", "dos", "portscan"]   # alphabetical = LabelEncoder order
ART_DIR.mkdir(parents=True, exist_ok=True)

def fuse(rf_labels, xgb_labels):
    """Vectorised version of app/routers/predict.py::make_verdict.
    agree -> that label | one says benign -> the other (threat) | two different threats -> RF."""
    rf_labels, xgb_labels = np.asarray(rf_labels), np.asarray(xgb_labels)
    out = rf_labels.copy()
    use_xgb = (rf_labels == "benign") & (xgb_labels != "benign")
    out[use_xgb] = xgb_labels[use_xgb]
    return out


def make_verdict(rf_label, xgb_label):          # reference implementation from the API
    if rf_label == xgb_label: return rf_label
    if rf_label == "benign": return xgb_label
    if xgb_label == "benign": return rf_label
    return rf_label

import xgboost as xgb_lib
from sklearn.metrics import f1_score, accuracy_score

S = joblib.load(WORK_DIR / "splits.joblib")
le = joblib.load(ART_DIR / "label_encoder.pkl")
scaler = joblib.load(ART_DIR / "scaler.pkl")
rf = joblib.load(ART_DIR / "rf_multiclass.pkl")
xgb = joblib.load(ART_DIR / "xgb_multiclass.pkl")
meta = json.loads((WORK_DIR / "data_meta.json").read_text())
FEATS = meta["features"]
Xte_s, Xte_raw, y_te = S["X_test"], S["X_test_raw"], S["y_test"]
y_true_all = le.inverse_transform(y_te)

N_ROB = 50_000          # rows used for the perturbation sweeps (None = all test rows)
N_SHAP_PER_CLASS = 400  # flows per class used for the SHAP analysis
rng = np.random.default_rng(0)
print("smoke run:", meta["split"]["smoke"])

## 1. TreeSHAP (exact) on XGBoost

In [ ]:
booster = xgb.get_booster()

def contribs_of(X):
    """returns array (n, n_classes, n_features + 1); last column is the bias term"""
    return booster.predict(xgb_lib.DMatrix(X), pred_contribs=True, validate_features=False)

# stratified sample of the test split
idx = np.concatenate([rng.choice(np.where(y_te == k)[0], min(N_SHAP_PER_CLASS, int((y_te == k).sum())), replace=False)
                      for k in range(len(CLASSES))])
Xs, ys = Xte_s[idx], y_te[idx]
C = contribs_of(Xs)
print("SHAP array:", C.shape)

# additivity: softmax(sum of contributions per class) must reproduce predict_proba
margins = C.sum(axis=2)
proba_shap = np.exp(margins - margins.max(1, keepdims=True)); proba_shap /= proba_shap.sum(1, keepdims=True)
print("max |softmax(SHAP sum) - predict_proba| =", float(np.abs(proba_shap - xgb.predict_proba(Xs)).max()))

In [ ]:
F = len(FEATS)
rows = {}
for k, c in enumerate(CLASSES):
    m = np.abs(C[ys == k, k, :F]).mean(axis=0)
    rows[c] = pd.Series(m, index=FEATS).sort_values(ascending=False).head(8)
top_tbl = pd.concat({c: pd.DataFrame({"feature": s.index, "mean|SHAP|": s.values.round(4)}).reset_index(drop=True)
                     for c, s in rows.items()}, axis=1)
display(top_tbl)
pd.concat([s.rename(c) for c, s in rows.items()], axis=1).to_csv(WORK_DIR / "shap_top_features.csv")

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
for ax, (c, s) in zip(axes, rows.items()):
    s.iloc[::-1].plot.barh(ax=ax, color="#2a6fbb"); ax.set_title(c); ax.tick_params(labelsize=7)
plt.tight_layout(); plt.savefig(WORK_DIR / "shap_top_features.png", dpi=200); plt.show()

## 2. Robustness sweeps

In [ ]:
sel = np.arange(len(y_te)) if N_ROB is None else rng.choice(len(y_te), min(N_ROB, len(y_te)), replace=False)
Xs_sel, Xraw_sel, y_sel = Xte_s[sel], Xte_raw[sel], y_true_all[sel]

def perturb_jitter(X_raw, eps): return X_raw * (1.0 + rng.uniform(-eps, eps, size=X_raw.shape))
def perturb_noise(X_s, sigma):  return X_s + rng.normal(0.0, sigma, size=X_s.shape)
def perturb_mask(X_s, p):       return np.where(rng.random(X_s.shape) < p, 0.0, X_s)   # scaled 0 == training median

def score_all(Xp):
    rf_l = le.inverse_transform(rf.predict(Xp)); xg_l = le.inverse_transform(xgb.predict(Xp))
    out = {}
    for name, lab in (("random_forest", rf_l), ("xgboost", xg_l), ("ensemble", fuse(rf_l, xg_l))):
        out[name] = {"accuracy": float(accuracy_score(y_sel, lab)),
                     "macro_f1": float(f1_score(y_sel, lab, labels=CLASSES, average="macro", zero_division=0))}
    return out

SWEEPS = {
    "raw_jitter_eps":       ([0.0, 0.05, 0.1, 0.2, 0.5],  "raw"),
    "gaussian_noise_sigma": ([0.0, 0.05, 0.1, 0.25, 0.5], "noise"),
    "feature_mask_prob":    ([0.0, 0.05, 0.1, 0.2, 0.3],  "mask"),
}
robust = {}
for name, (levels, kind) in SWEEPS.items():
    robust[name] = []
    for lv in levels:
        if kind == "raw":
            Xp = scaler.transform(pd.DataFrame(perturb_jitter(Xraw_sel, lv), columns=FEATS))
        elif kind == "noise":
            Xp = perturb_noise(Xs_sel, lv)
        else:
            Xp = perturb_mask(Xs_sel, lv)
        robust[name].append({"level": lv, **score_all(Xp)})
    print("done:", name)

tbl = pd.DataFrame([{"perturbation": n, "level": r["level"],
                     **{f"{m} macroF1": round(100 * r[m]["macro_f1"], 2) for m in ("random_forest", "xgboost", "ensemble")}}
                    for n, rs in robust.items() for r in rs])
display(tbl)

In [ ]:
titles = {"raw_jitter_eps": "Raw-space jitter (± fraction)", "gaussian_noise_sigma": "Gaussian noise σ (IQR units)",
          "feature_mask_prob": "Features masked (probability)"}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, (name, rs) in zip(axes, robust.items()):
    xs = [r["level"] for r in rs]
    for m, st in (("random_forest", "--"), ("xgboost", ":"), ("ensemble", "-")):
        ax.plot(xs, [100 * r[m]["macro_f1"] for r in rs], st, marker="o", label=m)
    ax.set_title(titles[name], fontsize=10); ax.set_xlabel("perturbation level"); ax.grid(alpha=.3)
axes[0].set_ylabel("macro F1 (%)"); axes[0].legend()
plt.tight_layout(); plt.savefig(WORK_DIR / "robustness.png", dpi=200); plt.show()

## 3. Explanation stability under noise and masking

In [ ]:
TOPK = 5
pred_clean = C.sum(axis=2).argmax(axis=1)                      # class predicted by XGBoost on clean input
top_clean = [set(np.argsort(-np.abs(C[i, pred_clean[i], :F]))[:TOPK]) for i in range(len(Xs))]

def stability(Xp):
    Cp = contribs_of(Xp)
    pred_p = Cp.sum(axis=2).argmax(axis=1)
    jac = []
    for i in range(len(Xs)):                                    # same class as on the clean input
        tp = set(np.argsort(-np.abs(Cp[i, pred_clean[i], :F]))[:TOPK])
        jac.append(len(top_clean[i] & tp) / len(top_clean[i] | tp))
    return {"mean_top5_jaccard": float(np.mean(jac)), "prediction_unchanged": float((pred_p == pred_clean).mean())}

stab = []
for kind, levels, fn in (("gaussian_noise_sigma", [0.05, 0.1, 0.25, 0.5], perturb_noise),
                         ("feature_mask_prob", [0.05, 0.1, 0.2, 0.3], perturb_mask)):
    for lv in levels:
        stab.append({"perturbation": kind, "level": lv, **stability(fn(Xs, lv))})
stab_df = pd.DataFrame(stab)
display(stab_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, (kind, g) in zip(axes, stab_df.groupby("perturbation", sort=False)):
    ax.plot(g["level"], g["mean_top5_jaccard"], marker="o", label="top-5 SHAP overlap (Jaccard)")
    ax.plot(g["level"], g["prediction_unchanged"], marker="s", ls="--", label="prediction unchanged")
    ax.set_title(titles[kind], fontsize=10); ax.set_xlabel("perturbation level"); ax.grid(alpha=.3); ax.set_ylim(0, 1.02)
axes[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(WORK_DIR / "shap_stability.png", dpi=200); plt.show()

## Save

In [ ]:
(WORK_DIR / "robustness_results.json").write_text(json.dumps(
    {"n_rows_used": int(len(sel)), "robustness": robust, "shap_stability": stab, "topk": TOPK,
     "smoke": meta["split"]["smoke"]}, indent=2), encoding="utf-8")
print("saved robustness_results.json, robustness.png, shap_stability.png, shap_top_features.csv/.png")
if meta["split"]["smoke"]: print("SMOKE run - do not use these results in the paper.")